# 🧪 Series: la columna con nombre propio — **SOLUCIÓN**

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/soluciones/03_series_solucion.ipynb)

**Objetivos:** crear Series, elegir entre `.loc`/`.iloc`, operar vectorizado y entender por qué pandas ALINEA por índice antes de sumar dos Series.

🔎 **Apóyate en el visualizador:** [DataFrames](https://mati3939.github.io/visualizador-numpy-pandas/#df) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: laboratorio clínico

Trabajas en un laboratorio que procesa exámenes de sangre. Cada resultado llega con el NOMBRE del paciente pegado — ni una tabla completa todavía, solo una columna con memoria. El mismo escenario del **Control 2** del curso.

## 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
pacientes = ['Ana', 'Beatriz', 'Carla', 'Diego', 'Elena',
             'Felipe', 'Gonzalo', 'Hilda', 'Ignacio', 'Javiera']
glucosa_valores = rng.integers(70, 180, size=len(pacientes))  # mg/dL en ayunas
glucosa_dict = dict(zip(pacientes, glucosa_valores))
print(glucosa_dict)

## 1 · Calentamiento

### 1.1 Tu primera Serie ⭐

Crea `glucosa`, una `pd.Series` a partir de `glucosa_dict` (el índice queda solo, con los nombres). Ponle nombre a la serie: `name='glucosa_mg_dl'`.

<details><summary>💡 Pista (haz clic)</summary>

`pd.Series(un_dict)` usa las LLAVES del dict como índice y los valores como datos, en el mismo orden en que se insertaron.

</details>

In [ ]:
glucosa = pd.Series(glucosa_dict, name='glucosa_mg_dl')  # el dict entrega el índice gratis

print(glucosa)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert isinstance(glucosa, pd.Series)
assert list(glucosa.index) == pacientes
assert (glucosa.values == glucosa_valores).all()
assert glucosa.name == 'glucosa_mg_dl'
print('✅ ¡Correcto!')

### 1.2 Por etiqueta o por posición ⭐

Obtén el valor de glucosa de `'Ana'` usando `.loc` (por NOMBRE) y guárdalo en `glucosa_ana`. Luego obtén el valor del TERCER paciente de la lista usando `.iloc` (por POSICIÓN) y guárdalo en `tercer_valor`.

<details><summary>💡 Pista (haz clic)</summary>

`.loc['Ana']` no sabe de posiciones; `.iloc[2]` no sabe de nombres — elige uno según lo que tengas a mano.

</details>

In [ ]:
glucosa_ana = glucosa.loc['Ana']    # .loc busca por ETIQUETA del índice
tercer_valor = glucosa.iloc[2]      # .iloc busca por POSICIÓN, parte de 0

print(glucosa_ana, tercer_valor)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert glucosa_ana == glucosa_dict['Ana']
assert tercer_valor == glucosa_valores[2]
print('✅ ¡Correcto!')

## 2 · Núcleo

### 2.1 Cambiar de unidades sin loops ⭐⭐

Convierte `glucosa` (mg/dL) a mmol/L en `glucosa_mmol`, dividiendo por `18.0182` (el factor de conversión clínico estándar).

In [ ]:
glucosa_mmol = glucosa / 18.0182   # la operación se aplica a CADA valor, el índice se conserva

print(glucosa_mmol.round(2))

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert glucosa_mmol.shape == glucosa.shape
assert list(glucosa_mmol.index) == list(glucosa.index)
assert abs(glucosa_mmol.loc['Ana'] - glucosa.loc['Ana'] / 18.0182) < 1e-9
print('✅ ¡Correcto!')

### 2.2 Un resumen estadístico ⭐⭐

Calcula `promedio` (la media de `glucosa`) y `resumen` (el resultado de `.describe()` sobre `glucosa`).

<details><summary>💡 Pista (haz clic)</summary>

`.describe()` te devuelve OTRA Serie: count, mean, std, min, cuartiles y max.

</details>

In [ ]:
promedio = glucosa.mean()
resumen = glucosa.describe()

print(promedio)
print(resumen)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert abs(promedio - sum(glucosa_valores) / len(glucosa_valores)) < 1e-9
assert 'mean' in resumen.index and 'std' in resumen.index
assert resumen.loc['count'] == len(glucosa)
print('✅ ¡Correcto!')

### 2.3 Filtrar pacientes con glucosa alta ⭐⭐

Guarda en `pacientes_altos` la Serie filtrada solo con los pacientes cuya glucosa supera 126 mg/dL (umbral clínico de diabetes en ayunas).

In [ ]:
pacientes_altos = glucosa[glucosa > 126]

print(pacientes_altos)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert (pacientes_altos > 126).all()
assert set(pacientes_altos.index).issubset(set(glucosa.index))
assert len(pacientes_altos) == (glucosa > 126).sum()
print('✅ ¡Correcto!')

### 2.4 Categorías con value_counts ⭐⭐

Crea `categoria`, una Serie de texto con el mismo índice que `glucosa`, que diga `'alto'` si supera 126, `'normal-alto'` si está entre 100 y 126, o `'normal'` si es menor. Luego cuenta cuántos pacientes hay en cada categoría con `.value_counts()` en `conteo`.

<details><summary>💡 Pista (haz clic)</summary>

`.value_counts()` cuenta cuántas veces se repite cada valor ÚNICO — perfecto para una columna de categorías.

</details>

In [ ]:
categoria = pd.Series(
    np.where(glucosa > 126, 'alto',
             np.where(glucosa > 100, 'normal-alto', 'normal')),
    index=glucosa.index)
conteo = categoria.value_counts()  # cuenta cuántas veces aparece cada valor

print(conteo)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert conteo.sum() == len(glucosa), 'la suma de las categorías debe dar el total de pacientes'
assert set(conteo.index).issubset({'alto', 'normal-alto', 'normal'})
assert conteo.get('alto', 0) == (glucosa > 126).sum()
print('✅ ¡Correcto!')

## 3 · Desafío: dos exámenes, dos índices distintos 🏁

In [ ]:
# un segundo examen, pero NO a los mismos pacientes: perdemos a los 2 primeros
# y se suman 2 pacientes nuevos que no salieron en glucosa
pacientes_trig = pacientes[2:] + ['Karla', 'Luis']
trigliceridos = pd.Series(rng.integers(80, 250, size=len(pacientes_trig)),
                          index=pacientes_trig, name='trigliceridos_mg_dl')
print(trigliceridos)

### 3.1 La alineación por índice ⭐ (ejercicio estrella) ⭐⭐⭐

Suma `glucosa + trigliceridos` en `suma`. Vas a ver `NaN` en algunos pacientes: en el comentario, explica POR QUÉ aparecen. Guarda además `n_nan`, cuántos `NaN` salieron.

<details><summary>💡 Pista (haz clic)</summary>

pandas suma POR ETIQUETA, no por posición: si un nombre no está en las dos Series, no tiene con qué sumarse y el resultado es `NaN`.

</details>

In [ ]:
suma = glucosa + trigliceridos
n_nan = suma.isnull().sum()

# aparecen NaN porque pandas ALINEA por índice antes de sumar: solo los
# pacientes que están en AMBAS Series obtienen una suma real. Los que están
# en una sola serie (Ana, Beatriz, Karla, Luis) no tienen con qué sumarse,
# y pandas rellena con NaN en vez de adivinar o botar la fila.
print(suma)
print('NaN:', n_nan)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
_esperado_nan = len(set(glucosa.index).symmetric_difference(set(trigliceridos.index)))
assert n_nan == _esperado_nan, 'cuenta los pacientes que NO están en ambas Series'
assert len(suma) == len(set(glucosa.index) | set(trigliceridos.index))
assert not suma.dropna().empty
print('✅ ¡Correcto!')

### 3.2 ¿Quién quedó fuera? 🏁 ⭐⭐⭐

Sin mirar el resultado de la suma, calcula directamente `solo_glucosa`: los pacientes que están en `glucosa` pero NO en `trigliceridos`, usando `.index.difference(...)`.

<details><summary>💡 Pista (haz clic)</summary>

`serieA.index.difference(serieB.index)` devuelve las etiquetas que están en A pero no en B — el mismo resultado que mirar los `NaN` del ejercicio anterior, pero calculado directo.

</details>

In [ ]:
solo_glucosa = glucosa.index.difference(trigliceridos.index)

print(solo_glucosa)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert set(solo_glucosa) == set(glucosa.index) - set(trigliceridos.index)
assert all(p not in trigliceridos.index for p in solo_glucosa)
print('✅ ¡Correcto!')

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **DataFrames**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).